In [ ]:
def main(datasources, start_date, end_date):
    """端到端 多频率 Transformer 极简示例。

    赛制约定: 平台只替换 datasources / start_date / end_date, 其中
    start_date~end_date 为【测试集区间】。训练区间写死(TRAIN_START/END),
    用样本外的测试区间做预测, 输出每日分数 ['date','instrument','score']。
    切勿用传入的 start_date/end_date 训练(数据泄漏, 会被审查)。

    端到端理念: 不做显式因子工程, 直接把原始 K 分钟量价/盘口序列当 token 序列
    喂给 Transformer。多频率(1m/5m/30m)各自一条编码分支, 在收盘决策点对齐后
    拼接送入回归头, 让模型同时看到微观(高频)与中观(低频)结构。
    仅用 10 个原始字段, 只做"按字段标准化 + 成交量 log"这类规则允许的预处理。
    """
    import time
    import numpy as np
    import pandas as pd
    import dai
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    import structlog

    logger = structlog.get_logger()

    # ---------- 配置 (写死, 不随平台入参变化) ----------
    TRAIN_START, TRAIN_END = "2023-01-01", "2023-06-30 23:59:59"
    SEQ_LEN = 64          # 每条分支回看多少个 bar (各频率共用, 覆盖时间随频率不同)
    EPOCHS, BATCH, LR, SEED = 5, 512, 1e-3, 42
    MAX_TRAIN_INSTRUMENTS = 200   # demo 限制训练标的数控制时长, 正式可放开
    FREQS = list(datasources.keys())   # 多频率分支, 顺序固定 (即 datasources 的键)

    PRICE_COLS = ["open", "high", "low", "close", "bid_price1", "ask_price1"]
    VOL_COLS   = ["volume", "amount", "bid_volume1", "ask_volume1"]  # 量纲大, 先 log1p
    FEATURE_COLS = PRICE_COLS + VOL_COLS
    N_FEAT = len(FEATURE_COLS)

    np.random.seed(SEED); torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info("运行设备", device=str(device), freqs=FREQS)

    # ---------- 模型: 每个频率一条 Transformer 分支, 池化后拼接 -> 回归头 ----------
    class FreqBranch(nn.Module):
        def __init__(self, n_feat, d_model, nhead=4, nlayers=2, dim_ff=128, seq_len=SEQ_LEN):
            super().__init__()
            self.proj = nn.Linear(n_feat, d_model)                  # 每个 bar -> token 向量
            self.pos = nn.Parameter(torch.zeros(1, seq_len, d_model))  # 可学习位置编码
            layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, 0.1,
                                               batch_first=True, activation="gelu")
            self.encoder = nn.TransformerEncoder(layer, nlayers)

        def forward(self, x):                                       # (B, L, N_FEAT) -> (B, d_model)
            return self.encoder(self.proj(x) + self.pos).mean(dim=1)

    class MultiFreqTransformer(nn.Module):
        def __init__(self, n_feat, freqs, d_model=64):
            super().__init__()
            self.freqs = freqs
            self.branches = nn.ModuleDict({f: FreqBranch(n_feat, d_model) for f in freqs})
            self.head = nn.Sequential(nn.LayerNorm(d_model * len(freqs)),
                                      nn.Linear(d_model * len(freqs), 1))

        def forward(self, xs):                                      # xs: 与 self.freqs 同序的张量列表
            v = torch.cat([self.branches[f](x) for f, x in zip(self.freqs, xs)], dim=1)
            return self.head(v).squeeze(-1)

    # ---------- 数据: 单频率切窗口, 以 (date,instrument) 为键便于跨频对齐 ----------
    def build_freq(table, sd, ed, instruments):
        """返回 windows={(date,ins): (SEQ_LEN,N_FEAT)} 和 labels={(date,ins): 未来1日收益}。"""
        t0 = time.time()
        buf = (pd.to_datetime(sd) - pd.Timedelta(days=20)).strftime("%Y-%m-%d")  # 缓冲凑回看窗口
        sql = f"SELECT date, instrument, {', '.join(FEATURE_COLS)} FROM {table} ORDER BY instrument, date"
        df = dai.query(sql, filters={"date": [buf, ed], "instrument": instruments}).df()
        df["instrument"] = df["instrument"].astype(str)
        df["date"] = pd.to_datetime(df["date"])
        for c in FEATURE_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        for c in VOL_COLS:
            df[c] = np.log1p(df[c].clip(lower=0))

        sd_ts, ed_ts = pd.to_datetime(sd), pd.to_datetime(ed)
        windows, labels = {}, {}
        for ins, sub in df.groupby("instrument", sort=False):
            sub = sub.sort_values("date")
            if len(sub) <= SEQ_LEN:
                continue
            feats = np.nan_to_num(sub[FEATURE_COLS].to_numpy(np.float32),
                                  nan=0.0, posinf=0.0, neginf=0.0)
            day = sub["date"].dt.normalize().to_numpy()
            close_pos = np.flatnonzero(np.append(day[1:] != day[:-1], True))  # 每日最后一根 bar
            close_px = sub["close"].to_numpy(np.float64)[close_pos]
            dates = day[close_pos]
            for k, p in enumerate(close_pos):
                d = pd.Timestamp(dates[k])
                if p + 1 < SEQ_LEN or d < sd_ts or d > ed_ts:
                    continue                                        # 历史不足 或 落在缓冲区
                key = (d, ins)
                windows[key] = feats[p - SEQ_LEN + 1: p + 1]
                if k + 1 < len(close_pos) and close_px[k] > 0:
                    r = close_px[k + 1] / close_px[k] - 1.0         # 未来 1 日收益
                    if np.isfinite(r):
                        labels[key] = np.float32(r)
        logger.info("单频构建完成", table=table, windows=len(windows),
                     elapsed=round(time.time() - t0, 2))
        return windows, labels

    def build_dataset(sd, ed, mode, instruments, stats=None):
        """跨频对齐: 只保留所有频率都凑齐窗口的 (date,instrument)。
        mode='train' 返回 (X,y,None,stats); 'infer' 返回 (X,None,idx_df,stats)。
        X 为 {freq: (N,SEQ_LEN,N_FEAT)}; stats 为各频 (mean,std), 推理复用。"""
        per = {f: build_freq(datasources[f], sd, ed, instruments) for f in FREQS}
        labels = per[FREQS[0]][1]                                   # 标签与频率无关, 取首个频率
        keys = set.intersection(*[set(per[f][0]) for f in FREQS])   # 各频率都有窗口的样本
        if mode == "train":
            keys &= set(labels)
        keys = sorted(keys)                                         # 按 (date,instrument) 对齐排序
        if not keys:
            raise RuntimeError(f"build_dataset 无样本 (mode={mode}, {sd}~{ed})")

        new_stats, X = {}, {}
        for f in FREQS:
            arr = np.stack([per[f][0][k] for k in keys]).astype(np.float32)
            if stats is None:                                       # 训练集上算, 推理复用
                flat = arr.reshape(-1, N_FEAT)
                new_stats[f] = (flat.mean(0).astype(np.float32), flat.std(0).astype(np.float32) + 1e-6)
            m, s = (new_stats if stats is None else stats)[f]
            X[f] = ((arr - m) / s).astype(np.float32)               # 按字段标准化
        stats = new_stats if stats is None else stats
        logger.info(f"{mode} 集构建完成", samples=len(keys))
        if mode == "train":
            return X, np.array([labels[k] for k in keys], np.float32), None, stats
        return X, None, pd.DataFrame(keys, columns=["date", "instrument"]), stats

    def pool(sd, ed):
        """区间内中证 1000 成分股代码 (查询过滤 & 对齐口径)。"""
        df = dai.query("SELECT DISTINCT instrument FROM bigalpha_2026_instruments",
                       filters={"date": [sd, ed]}).df()
        return df["instrument"].astype(str).tolist()

    # ---------- 训练 (写死训练区间, 从零训练) ----------
    logger.info("构建训练集", start=TRAIN_START, end=TRAIN_END)
    Xtr, ytr, _, stats = build_dataset(
        TRAIN_START, TRAIN_END, "train", pool(TRAIN_START, TRAIN_END)[:MAX_TRAIN_INSTRUMENTS])
    lo, hi = np.percentile(ytr, [1, 99]); ytr = np.clip(ytr, lo, hi)  # winsorize 标签

    model = MultiFreqTransformer(N_FEAT, FREQS).to(device)
    logger.info("可训练参数量", n_params=sum(p.numel() for p in model.parameters()))
    tensors = [torch.from_numpy(Xtr[f]) for f in FREQS] + [torch.from_numpy(ytr)]
    loader = DataLoader(TensorDataset(*tensors), batch_size=BATCH, shuffle=True,
                        pin_memory=(device.type == "cuda"))
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    model.train()
    for ep in range(EPOCHS):
        t, tot, nb = time.time(), 0.0, 0
        for batch in loader:
            *xs, yb = batch
            xs = [x.to(device, non_blocking=True) for x in xs]
            yb = yb.to(device, non_blocking=True)
            opt.zero_grad()
            loss = loss_fn(model(xs), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item(); nb += 1
        logger.info("epoch 完成", epoch=ep + 1, mse=round(tot / max(nb, 1), 8),
                    elapsed=round(time.time() - t, 2))

    # ---------- 推理 (样本外测试区间) ----------
    logger.info("构建测试集并预测", start=str(start_date), end=str(end_date))
    Xte, _, idx_df, _ = build_dataset(start_date, end_date, "infer", pool(start_date, end_date), stats)
    model.eval()
    preds = []
    xts = [torch.from_numpy(Xte[f]) for f in FREQS]
    with torch.no_grad():
        for i in range(0, len(idx_df), BATCH):
            xs = [x[i:i + BATCH].to(device) for x in xts]
            preds.append(model(xs).cpu().numpy())
    idx_df["score"] = np.concatenate(preds).astype(np.float64)

    # ---------- 对齐中证 1000 + 规范输出 ----------
    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={"date": [start_date, end_date]}).df()
    stk["instrument"] = stk["instrument"].astype(str)
    stk["date"] = pd.to_datetime(stk["date"]).dt.normalize()
    idx_df["date"] = pd.to_datetime(idx_df["date"]).dt.normalize()
    result = (pd.merge(idx_df, stk, on=["date", "instrument"], how="inner")
                .replace([np.inf, -np.inf], np.nan).dropna(subset=["score"])
                .drop_duplicates(["date", "instrument"])[["date", "instrument", "score"]]
                .reset_index(drop=True))
    logger.info("分数构建完成", rows=len(result), days=result["date"].nunique(),
                instruments=result["instrument"].nunique())
    return result


if __name__ == "__main__":
    from bigmodule import M
    import structlog

    logger = structlog.get_logger()

    # 定死多频率数据集 (system 注入逻辑暂不处理)
    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "bar5m": "bigalpha_2026_stock_bar5m",
        "bar30m": "bigalpha_2026_stock_bar30m",
    }

    # 本地用一小段区间模拟「平台注入的测试集区间」(训练区间已在 main 内写死)
    start_date, end_date = "2024-01-01 00:00:00", "2024-02-29 23:59:59"
    logger.info("计算分数", start=start_date, end=end_date)
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())

    # 评估系统: 分数经风格剔除后等价于每日单因子, show=True 画绩效图 (IC / 分组 / 压力期)
    logger.info("开始评估分数")
    M.bigalpah_e2emodel(score_data=score_data, show=True)
